# GraphRAG Knowledge Graph với FalkorDB
**Data Mining Miniproject — Demo từng bước**

Pipeline:
1. ETL: Amazon Reviews → FalkorDB
2. Weighted Louvain Community Detection
3. GraphRAG: Cypher context + LLM answer
4. Evaluation: P@K, NDCG@K so sánh với SVD baseline

In [ ]:
import sys, os
sys.path.insert(0, '../src')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

# Nếu dùng OpenAI, set key ở đây
# os.environ['OPENAI_API_KEY'] = 'sk-...'

print('Setup hoàn tất!')

## Bước 1: ETL — Load dữ liệu vào FalkorDB

In [ ]:
from etl import download_dataset, load_reviews, GraphBuilder, cosine_similarity_products

# Tải dataset (chỉ cần chạy lần đầu)
download_dataset()

# Load và filter reviews
df = load_reviews(max_rows=30_000)
print(f'Loaded: {len(df):,} reviews')
df.head()

In [ ]:
# Build Knowledge Graph
builder = GraphBuilder()
builder.clear()
builder.create_indexes()
builder.load_users(df)
builder.load_products(df)
builder.load_categories(df)
builder.load_reviews(df)
builder.add_user_stats(df)

# Product similarity edges
sim_edges = cosine_similarity_products(df, top_k=5)
builder.load_similar(sim_edges)
builder.summary()

In [ ]:
# Visualize distribution
fig, axes = plt.subplots(1, 3, figsize=(13, 4))

# Rating distribution
df['rating'].value_counts().sort_index().plot(
    kind='bar', ax=axes[0], color='#534AB7', alpha=0.85
)
axes[0].set_title('Rating Distribution')
axes[0].set_xlabel('Rating')
axes[0].spines[['top','right']].set_visible(False)

# Reviews per user
user_counts = df['user_id'].value_counts()
axes[1].hist(user_counts.values, bins=30, color='#1D9E75', alpha=0.85, edgecolor='none')
axes[1].set_title('Reviews per User')
axes[1].set_xlabel('Review count')
axes[1].spines[['top','right']].set_visible(False)

# Top categories
df['category'].value_counts().head(8).plot(
    kind='barh', ax=axes[2], color='#BA7517', alpha=0.85
)
axes[2].set_title('Top Categories')
axes[2].spines[['top','right']].set_visible(False)

plt.tight_layout()
plt.show()

## Bước 2: Weighted Louvain Community Detection

In [ ]:
from clustering import WeightedCommunityDetector

detector  = WeightedCommunityDetector()
partition = detector.run(resolution=1.0)

print(f'\nTổng communities: {len(set(partition.values()))}')
print(f'Tổng users assigned: {len(partition)}')

In [ ]:
# So sánh Louvain vs K-means modularity
import community as community_louvain

G             = detector._graph_G
n_comm        = len(set(partition.values()))
km_partition  = detector.kmeans_baseline(detector.fetch_user_ratings(), n_clusters=n_comm)

louvain_mod = community_louvain.modularity(partition,    G, weight='weight')
kmeans_mod  = community_louvain.modularity(km_partition, G, weight='weight')

fig, ax = plt.subplots(figsize=(6, 4))
bars = ax.bar(['Weighted Louvain\n(proposed)', 'K-means\n(baseline)'],
              [louvain_mod, kmeans_mod],
              color=['#534AB7', '#888780'], alpha=0.85, width=0.5)
for bar, val in zip(bars, [louvain_mod, kmeans_mod]):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.002,
            f'{val:.4f}', ha='center', fontsize=12, fontweight='bold')
ax.set_title('Modularity Score Comparison', fontsize=13)
ax.set_ylabel('Modularity')
ax.set_ylim(0, max(louvain_mod, kmeans_mod) * 1.2)
ax.spines[['top','right']].set_visible(False)
plt.tight_layout()
plt.show()
print(f'Improvement: +{(louvain_mod - kmeans_mod):.4f} modularity')

## Bước 3: GraphRAG — Cypher context + LLM answer

In [ ]:
from graphrag import GraphRAGPipeline

rag = GraphRAGPipeline()

# Lấy sample user từ graph
from falkordb import FalkorDB
db    = FalkorDB(host='localhost', port=6379)
graph = db.select_graph('amazon_graph')
res   = graph.query('MATCH (u:User) WHERE u.community_id IS NOT NULL RETURN u.user_id LIMIT 1')
sample_user = res.result_set[0][0]
print(f'Demo user: {sample_user}')

In [ ]:
question = 'Gợi ý tai nghe không dây tốt cho tôi'

result = rag.recommend(sample_user, question)

print('=== GraphRAG Answer ===')
print(result['answer'])
print(f'\nCommunity: {result["community_id"]}')
print(f'Candidates: {len(result["candidates"])} products')

In [ ]:
# Xem context được gửi cho LLM
print('=== Context gửi cho LLM ===')
print(result['context'])

In [ ]:
# Top candidates từ graph
pd.DataFrame(result['candidates']).head(10)

## Bước 4: Evaluation — P@K, NDCG@K, HR@K

In [ ]:
from evaluation import Evaluator

# Load partition từ graph
res2      = graph.query('MATCH (u:User) WHERE u.community_id IS NOT NULL RETURN u.user_id, u.community_id')
partition = {r[0]: int(r[1]) for r in res2.result_set}

ev      = Evaluator(df, partition)
summary = ev.evaluate(k_values=[5, 10, 20], sample_n=300)
ev.print_results(summary)

In [ ]:
fig = ev.plot_results(summary)
plt.show()

## Cypher Explorer — Các query hữu ích

Bạn có thể chạy các query Cypher trực tiếp để khám phá graph.

In [ ]:
# Top sản phẩm bán chạy nhất
res = graph.query('''
    MATCH (u:User)-[b:BOUGHT]->(p:Product)
    WITH p, count(b) AS cnt, avg(b.weight) AS avg_r
    RETURN p.name AS name, p.category AS cat,
           cnt AS purchases, round(avg_r*100)/100 AS avg_rating
    ORDER BY cnt DESC LIMIT 10
''')
pd.DataFrame(res.result_set, columns=['Name','Category','Purchases','Avg Rating'])

In [ ]:
# Community overview
res = graph.query('''
    MATCH (u:User) WHERE u.community_id IS NOT NULL
    RETURN u.community_id AS cid,
           count(u) AS users,
           round(avg(u.avg_rating)*100)/100 AS avg_rating,
           round(avg(u.purchase_count)*10)/10 AS avg_purchases
    ORDER BY users DESC LIMIT 15
''')
pd.DataFrame(res.result_set, columns=['Community','Users','Avg Rating','Avg Purchases'])

In [ ]:
# Similar products cho một sản phẩm cụ thể
sample_pid = df['product_id'].iloc[0]
res = graph.query('''
    MATCH (p:Product {product_id: $pid})-[s:SIMILAR]->(q:Product)
    RETURN p.name AS source, q.name AS similar_to, s.weight AS similarity
    ORDER BY s.weight DESC
''', {'pid': sample_pid})
pd.DataFrame(res.result_set, columns=['Source','Similar To','Similarity'])